# Picturebook I2V — LTX-Video 2B Distilled
Drive上の元絵をfirst-frame conditionにして、無料Colab GPUで短いI2Vを生成します。


In [ ]:
import torch, os
from google.colab import drive
assert torch.cuda.is_available(), "GPUランタイムが有効ではありません"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3,1))
print("CC:", torch.cuda.get_device_capability(0))
drive.mount('/content/drive')
INPUT='/content/drive/MyDrive/panel04_ref.png'
assert os.path.exists(INPUT), INPUT
print("INPUT OK:", INPUT)


In [ ]:
%cd /content
!rm -rf LTX-Video
!git clone -q https://github.com/Lightricks/LTX-Video.git
%cd /content/LTX-Video
!pip -q install -e ".[inference]"
!pip -q install imageio[ffmpeg]
print("installed")


In [ ]:
from pathlib import Path
import torch, yaml, re, os, subprocess, textwrap
repo=Path('/content/LTX-Video')
src_inf=repo/'ltx_video/inference.py'
src_pipe=repo/'ltx_video/pipelines/pipeline_ltx_video.py'
cfg= yaml.safe_load((repo/'configs/ltxv-2b-0.9.8-distilled.yaml').read_text())
cfg['prompt_enhancement_words_threshold']=0
major,_=torch.cuda.get_device_capability(0)
if major < 8:
    cfg['precision']='mixed_precision'
    for p in (src_inf,src_pipe):
        t=p.read_text().replace('torch.bfloat16','torch.float16')
        p.write_text(t)
    t=src_inf.read_text()
    t=t.replace('return Transformer3DModel.from_pretrained(ckpt_path)\n',
                'return Transformer3DModel.from_pretrained(ckpt_path).to(torch.float16)\n')
    src_inf.write_text(t)
    print('Applied T4/legacy GPU FP16 compatibility patch')
else:
    print('BF16-capable GPU')
cfg_path=Path('/content/ltx2b_colab.yaml')
cfg_path.write_text(yaml.safe_dump(cfg,sort_keys=False))
print(cfg_path.read_text())


In [ ]:
PROMPT="Preserve the exact original hand-painted gouache storybook illustration, character design, facial proportions, color palette, grain, brush texture, flowers, clouds, and blue-and-white bird. Do not redesign or restyle anything. Animate the existing scene as if filmed in a cinematic gust of wind. The blue-and-white bird beats its wings and flies dynamically from left to right, passing slightly closer to camera before climbing away. The girl's eyes naturally follow the bird, her hand lifts slightly, her dark hair flows with delayed follow-through. Flowers, grasses, and loose blue leaves bend and sweep left to right in a visible wave. A few foreground leaves pass close to the lens for depth. Use a subtle cinematic push-in and slight rightward tracking camera. Keep the scene whimsical, warm, child-friendly, painterly, coherent, and faithful to the first frame. No morphing, no new characters, no costume changes, no photorealism."
NEGATIVE='worst quality, inconsistent motion, blurry, jittery, distorted anatomy, duplicated objects, new characters, photorealistic, style change, color shift, warped face, extra limbs, melting shapes'
OUT='/content/drive/MyDrive/Picturebook-I2V'
os.makedirs(OUT,exist_ok=True)


In [ ]:
import subprocess, glob, os, torch, gc, shlex, pathlib, sys
def run(width,height,frames):
    cmd=[
      'python','inference.py',
      '--prompt',PROMPT,
      '--negative_prompt',NEGATIVE,
      '--conditioning_media_paths',INPUT,
      '--conditioning_strengths','1.0',
      '--conditioning_start_frames','0',
      '--height',str(height),'--width',str(width),
      '--num_frames',str(frames),'--frame_rate','24',
      '--seed','43021',
      '--pipeline_config','/content/ltx2b_colab.yaml',
      '--offload_to_cpu','True',
      '--image_cond_noise_scale','0.08',
      '--output_path',OUT,
    ]
    print('RUN:', ' '.join(shlex.quote(x) for x in cmd))
    subprocess.run(cmd,check=True)
try:
    run(512,288,81)
except subprocess.CalledProcessError:
    print('First attempt failed; retrying smaller resolution')
    gc.collect(); torch.cuda.empty_cache()
    run(448,256,65)
videos=sorted(glob.glob(OUT+'/*.mp4'), key=os.path.getmtime)
assert videos, 'No MP4 generated'
VIDEO=videos[-1]
print('VIDEO:',VIDEO, os.path.getsize(VIDEO))


In [ ]:
from IPython.display import Video, display
display(Video(VIDEO, embed=True))
